In [ ]:
import os
import glob
import torch
import random
import pandas as pd
from PIL import Image
import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

from transformers import ViTFeatureExtractor, ViTModel
from patches import get_image_patches
from embedding import get_embedding, uuid_from_dict

model_id = "google/vit-base-patch16-224-in21k"

mps_device = torch.device("mps")

feature_extractor = ViTFeatureExtractor.from_pretrained(model_id)
model = ViTModel.from_pretrained(model_id).to(mps_device)
model.eval()

In [ ]:
client = QdrantClient(location="http://localhost:6333", port=None, grpc_port=None, timeout=600)
client.create_collection(
    collection_name="vit-base-patch16",
    vectors_config=VectorParams(size=768, distance=Distance.COSINE)
)

In [ ]:
patch_size = 224
window_shift = 56

source_pattern = os.path.join("..", "..", "data", "artwork", "**")
images = []
patch_description = []

patch_generator = get_image_patches(
    source_pattern, 
    patch_size, 
    window_shift, 
    batch_size=512,
    skip=None,
    zoom=1
)

embedding_generator = get_embedding(
    patch_generator, 
    mps_device, 
    model,
    feature_extractor
)

In [ ]:
for uuids, infos, X in embedding_generator:

    for uuid, info in zip(uuids, infos):
        info.update({"uuid": uuid})

    client.upsert(
        collection_name="vit-base-patch16",
        points=[
            PointStruct(
                    id=uuid_from_dict(info),
                    vector=Xi,
                    payload=info
            )
            for uuid, info, Xi in zip(uuids, infos, X)
            ]
    )
